# Credit-Risk Feature Selection via Repeated-Seed SHAP Stability

**Question.** Can SHAP-rank stability identify a compact, reproducible feature subset that retains
predictive discrimination across RF, XGBoost and LightGBM?

**Protocol**
1. Merge bureau + bank data, build the binary target (`P4` = 1), make one fixed stratified split.
2. Preprocessing, feature selection, tuning and model fitting use the **development set only**. The test set is reserved for the final evaluation and one exploratory Credit_Score diagnostic (both in §7); the diagnostic feeds nothing back into fitting or selection.
3. Two branches share a single code path: `NO_CS` (Credit_Score removed, primary evidence) and `WITH_CS` (original experiment, saturated by Credit_Score).
4. Feature sets: **F0** all · **F1** mutual-information top-k · **F2** SHAP top-k · **F3** stable SHAP (mean rank ≤ 10 and rank range ≤ 3 over 100 seeds).

## 1. Configuration

In [1]:
import json, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub, shap, sklearn, xgboost, lightgbm
from IPython.display import display
from joblib import Parallel, delayed
from lightgbm import LGBMClassifier
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- data ----------------------------------------------------------------------------------
# Download latest version of the dataset directly
DATA_DIR = kagglehub.dataset_download("saurabhbadole/leading-indian-bank-and-cibil-real-world-dataset")
print("Path to dataset files:", DATA_DIR)
TARGET, POSITIVE_CLASS, ID_COL = "Approved_Flag", "P4", "PROSPECTID"
PROTECTED = ["GENDER", "MARITALSTATUS"]                 # structurally excluded from predictors
SENTINEL = -99999
EXPECTED_ROWS, EXPECTED_COLS = 51_336, 87

# ---- protocol ------------------------------------------------------------------------------
SPLIT_SEED, SEED, TEST_SIZE = 42, 2026, 0.20
TOP_K = 10                                              # size of F1 / F2
N_SEEDS = 100                                           # SHAP-stability replicates
MEAN_RANK_MAX, RANK_RANGE_MAX = 10, 3                   # stability rule for F3
SHAP_SAMPLE, STABILITY_SAMPLE = 2048, 512               # dev rows used to explain models
THRESHOLD = 0.40                                        # single decision threshold for all threshold metrics
N_BOOT = 5000
N_JOBS = -1
RUN_TUNING = False                                      # True re-runs the 48-feature search behind the frozen params
RUN_WITH_CREDIT_SCORE_BRANCH = True
MI_DISCRETE_MASK = False   # True: treat one-hot / missing-indicator columns as discrete in MI (sensitivity check).
                           # The frozen results used False (all columns continuous).
USE_FROZEN_SETS = True     # True: §7 evaluates the frozen feature lists below (reproduces the official experiment);
                           # the re-derived lists are still computed and compared in §9.

OUT = Path("phase6_optimized_outputs"); OUT.mkdir(exist_ok=True)   # separate from the frozen phase6_outputs/

# Fields flagged by the project's earlier target-proxy audit (prior evidence, NOT proof of leakage).
AUDITED_PROXY_COLUMNS = [
    "Credit_Score", "CC_enq", "CC_enq_L12m", "CC_enq_L6m", "PL_enq", "PL_enq_L12m", "PL_enq_L6m",
    "Tot_Missed_Pmnt", "enq_L12m", "enq_L3m", "enq_L6m", "first_prod_enq2", "last_prod_enq2",
    "max_deliq_12mts", "max_deliq_6mts", "max_delinquency_level", "max_recent_level_of_deliq",
    "num_deliq_12mts", "num_deliq_6_12mts", "num_deliq_6mts", "num_std", "num_std_12mts", "num_std_6mts",
    "num_times_30p_dpd", "num_times_60p_dpd", "num_times_delinquent", "pct_CC_enq_L6m_of_L12m",
    "pct_CC_enq_L6m_of_ever", "pct_PL_enq_L6m_of_L12m", "pct_PL_enq_L6m_of_ever", "recent_level_of_deliq",
    "time_since_first_deliquency", "time_since_recent_deliquency", "time_since_recent_enq", "tot_enq",
]
assert len(AUDITED_PROXY_COLUMNS) == len(set(AUDITED_PROXY_COLUMNS)) == 35

# Feature sets frozen in the project hand-off (used for evaluation when USE_FROZEN_SETS=True, and for the §9 comparison).
FROZEN_SETS = {
    "NO_CS": {
        "F1_MI": ["enq_L3m", "enq_L6m", "enq_L12m", "time_since_recent_enq", "PL_enq_L6m", "tot_enq",
                  "pct_PL_enq_L6m_of_ever", "PL_enq_L12m", "pct_PL_enq_L6m_of_L12m", "PL_enq"],
        "F2_SHAP": ["enq_L3m", "Age_Oldest_TL", "num_times_delinquent", "time_since_recent_enq",
                    "time_since_recent_deliquency", "pct_PL_enq_L6m_of_L12m", "num_std_12mts",
                    "pct_PL_enq_L6m_of_ever", "max_deliq_12mts", "max_recent_level_of_deliq"],
        "F3_STABLE_SHAP": ["enq_L3m", "Age_Oldest_TL", "num_times_delinquent", "time_since_recent_enq",
                           "time_since_recent_deliquency", "pct_PL_enq_L6m_of_L12m", "pct_PL_enq_L6m_of_ever",
                           "num_std_12mts", "max_deliq_12mts"],
    },
    "WITH_CS": {   # F1 was not recorded in the hand-off, so it is always re-derived
        "F2_SHAP": ["Credit_Score", "enq_L3m", "num_times_delinquent", "Age_Oldest_TL", "time_since_recent_enq",
                    "time_since_recent_deliquency", "pct_PL_enq_L6m_of_L12m", "pct_PL_enq_L6m_of_ever",
                    "max_recent_level_of_deliq", "num_std_12mts"],
        "F3_STABLE_SHAP": ["Credit_Score", "enq_L3m", "num_times_delinquent", "time_since_recent_enq", "Age_Oldest_TL"],
    },
}

Path to dataset files: /kaggle/input/datasets/saurabhbadole/leading-indian-bank-and-cibil-real-world-dataset


## 2. Data, target and development / test split

In [2]:
def read_excel(name):
    matches = sorted(Path(DATA_DIR).rglob(name))
    assert matches, f"{name} not found under {DATA_DIR}"
    return pd.read_excel(matches[0])

external, internal = read_excel("External_Cibil_Dataset.xlsx"), read_excel("Internal_Bank_Dataset.xlsx")
for name, part in (("external", external), ("internal", internal)):
    assert part[ID_COL].is_unique and part[ID_COL].notna().all(), f"{name}: {ID_COL} not unique / has nulls"

df = external.merge(internal, on=ID_COL, how="inner", validate="one_to_one")
assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), f"unexpected merged shape {df.shape}"
assert df[TARGET].notna().all()

y = (df[TARGET] == POSITIVE_CLASS).astype(int)
X = df.drop(columns=[ID_COL, TARGET, *PROTECTED])
RAW_FEATURES = X.columns.tolist()
assert len(RAW_FEATURES) == 83 and set(AUDITED_PROXY_COLUMNS) <= set(RAW_FEATURES)

# Sentinel -99999 -> NaN (a fixed value substitution, so doing it before the split leaks nothing).
sentinel = (X == SENTINEL).sum()
sentinel = sentinel[sentinel > 0]
print(f"{len(sentinel)} columns contain {SENTINEL}  ({sentinel.sum():,} occurrences)")
X = X.replace(SENTINEL, np.nan)
assert not (X == SENTINEL).any().any()

print("\nApproved_Flag:", df[TARGET].value_counts().sort_index().to_dict())
print(f"Positive rate ({POSITIVE_CLASS}): {y.mean():.4%}")

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SPLIT_SEED)
assert X_dev.index.isin(X_test.index).sum() == 0

print(f"\nDevelopment {X_dev.shape}  positive rate {y_dev.mean():.4%}")
print(f"Test        {X_test.shape}  positive rate {y_test.mean():.4%}")

json.dump({"development_indices": X_dev.index.tolist(), "test_indices": X_test.index.tolist()},
          open(OUT / "split_indices.json", "w"))

23 columns contain -99999  (320,792 occurrences)

Approved_Flag: {'P1': 5803, 'P2': 32199, 'P3': 7452, 'P4': 5882}
Positive rate (P4): 11.4578%

Development (41068, 83)  positive rate 11.4590%
Test        (10268, 83)  positive rate 11.4531%


## 3. Shared helpers (preprocessing, feature mapping, models, metrics)

In [3]:
def make_preprocessor(frame):
    """Median-impute (+ missing indicators) and standardise numerics; mode-impute and one-hot categoricals."""
    num = frame.select_dtypes(include=[np.number]).columns.tolist()
    cat = frame.select_dtypes(exclude=[np.number]).columns.tolist()
    parts = []
    if num:
        parts.append(("num", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                                       ("scaler", StandardScaler())]), num))
    if cat:
        parts.append(("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                       ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat))
    return ColumnTransformer(parts, remainder="drop")


def encoded_to_raw(pre):
    """Map every encoded column of a FITTED preprocessor to its parent raw feature."""
    mapping = {}
    for name, pipe, cols in pre.transformers_:
        if name == "num":      # imputed columns + their missing-indicator columns
            for n in pipe.named_steps["imputer"].get_feature_names_out(cols):
                mapping[f"num__{n}"] = n.removeprefix("missingindicator_")
        elif name == "cat":    # one-hot descendants
            ohe = pipe.named_steps["onehot"]
            parents = np.repeat(cols, [len(c) for c in ohe.categories_])
            for n, parent in zip(ohe.get_feature_names_out(cols), parents):
                mapping[f"cat__{n}"] = parent
    assert set(mapping) == set(pre.get_feature_names_out()), "encoded -> raw mapping is incomplete"
    return mapping


def to_raw(values, names, mapping):
    """Sum per-encoded-column scores into per-raw-feature scores."""
    return pd.Series(values, index=[mapping[n] for n in names]).groupby(level=0).sum()


def mean_abs_shap(model, sample):
    sv = shap.TreeExplainer(model).shap_values(sample)
    sv = np.asarray(sv[-1] if isinstance(sv, list) else sv)
    assert sv.ndim == 2
    return np.abs(sv).mean(axis=0)


# ---- models (frozen configuration) ----------------------------------------------------------
SPW = (y_dev == 0).sum() / (y_dev == 1).sum()           # class weight from development data only
MODEL_CLASSES = {"RF": RandomForestClassifier, "XGB": XGBClassifier, "LGBM": LGBMClassifier}
MODEL_PARAMS = {
    "RF":   dict(n_estimators=600, min_samples_leaf=20, max_features=0.5, max_depth=None, class_weight="balanced"),
    "XGB":  dict(n_estimators=300, max_depth=5, learning_rate=0.03, subsample=0.8, colsample_bytree=1.0,
                 scale_pos_weight=SPW, objective="binary:logistic", eval_metric="logloss"),
    "LGBM": dict(n_estimators=150, num_leaves=31, learning_rate=0.03, subsample=1.0, colsample_bytree=1.0,
                 scale_pos_weight=SPW, objective="binary", verbosity=-1),
}
XGB_SELECTOR_PARAMS = dict(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                           scale_pos_weight=SPW, objective="binary:logistic", eval_metric="logloss")

def make_model(name):
    return MODEL_CLASSES[name](**MODEL_PARAMS[name], random_state=SEED, n_jobs=-1)


# ---- metrics: one implementation, used for the final evaluation and for the bootstrap -------
def confusion(y_true, y_prob, thr=THRESHOLD):
    pred, pos = y_prob >= thr, y_true == 1
    tp, fp, fn = int((pred & pos).sum()), int((pred & ~pos).sum()), int((~pred & pos).sum())
    return len(y_true) - tp - fp - fn, fp, fn, tp

def f_beta(tp, fp, fn, beta):
    b2 = beta ** 2
    denom = (1 + b2) * tp + b2 * fn + fp
    return (1 + b2) * tp / denom if denom else 0.0

def core_metrics(y_true, y_prob):
    """The four metrics that are bootstrapped."""
    _, fp, fn, tp = confusion(y_true, y_prob)
    return {"ROC_AUC": roc_auc_score(y_true, y_prob), "PR_AUC": average_precision_score(y_true, y_prob),
            "Recall": tp / (tp + fn) if tp + fn else 0.0, "F2": f_beta(tp, fp, fn, 2)}

def full_metrics(y_true, y_prob):
    tn, fp, fn, tp = confusion(y_true, y_prob)
    return {**core_metrics(y_true, y_prob),
            "Precision": tp / (tp + fp) if tp + fp else 0.0, "F1": f_beta(tp, fp, fn, 1),
            "KS": ks_2samp(y_prob[y_true == 1], y_prob[y_true == 0]).statistic,
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

## 4. Feature audit and Credit_Score diagnostic (development data only)

Univariate proxy-AUC is a **diagnostic**. It does not label a feature valid or invalid: the supplied files contain no
provenance / time-of-availability metadata, so no independent validity gate can be applied.

In [4]:
def proxy_auc(feature_frame, target):
    """Direction-agnostic out-of-fold AUC of a depth-3 tree on a single raw feature."""
    numeric = pd.api.types.is_numeric_dtype(feature_frame.iloc[:, 0])
    steps = [SimpleImputer(strategy="median" if numeric else "most_frequent")]
    if not numeric:
        steps.append(OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    est = make_pipeline(*steps, DecisionTreeClassifier(max_depth=3, random_state=SEED))
    cv = StratifiedKFold(3, shuffle=True, random_state=SEED)
    oof = cross_val_predict(est, feature_frame, target, cv=cv, method="predict_proba")[:, 1]
    auc = roc_auc_score(target, oof)
    return max(auc, 1 - auc)

feature_audit = pd.DataFrame({
    "raw_feature": RAW_FEATURES,
    "dtype": X_dev.dtypes.astype(str).values,
    "missing_rate": X_dev.isna().mean().values,
    "unique_values": X_dev.nunique().values,
    "proxy_auc": Parallel(n_jobs=N_JOBS)(delayed(proxy_auc)(X_dev[[f]], y_dev) for f in RAW_FEATURES),
})
feature_audit["proxy_review_flag"] = feature_audit["proxy_auc"] >= 0.90
feature_audit["previous_audit_flag"] = feature_audit["raw_feature"].isin(AUDITED_PROXY_COLUMNS)
feature_audit = feature_audit.sort_values("proxy_auc", ascending=False).reset_index(drop=True)

feature_audit.to_csv(OUT / "phase2_feature_audit.csv", index=False)
feature_audit.to_json(OUT / "phase2_feature_audit.json", orient="records", indent=2)

print("previously audited:", feature_audit["previous_audit_flag"].sum(), "| not audited:", (~feature_audit["previous_audit_flag"]).sum())
print("\nproxy-AUC >= 0.90  vs  previous audit flag:")
print(pd.crosstab(feature_audit["proxy_review_flag"], feature_audit["previous_audit_flag"]))
display(feature_audit.head(15))

previously audited: 35 | not audited: 48

proxy-AUC >= 0.90  vs  previous audit flag:
previous_audit_flag  False  True 
proxy_review_flag                
False                   48     34
True                     0      1


,raw_feature,dtype,missing_rate,unique_values,proxy_auc,proxy_review_flag,previous_audit_flag
0,Credit_Score,int64,0.000000,240,1.000000,True,True
1,enq_L3m,float64,0.123381,35,0.875435,False,True
2,enq_L6m,float64,0.123381,52,0.854244,False,True
3,enq_L12m,float64,0.123381,64,0.818486,False,True
4,time_since_recent_enq,float64,0.123381,2232,0.787809,False,True
5,tot_enq,float64,0.123381,82,0.768848,False,True
6,PL_enq_L6m,float64,0.123381,27,0.746170,False,True
7,pct_PL_enq_L6m_of_ever,float64,0.000000,129,0.741367,False,True
8,pct_PL_enq_L6m_of_L12m,float64,0.000000,83,0.738122,False,True
9,PL_enq_L12m,float64,0.123381,34,0.733932,False,True


In [5]:
# Credit_Score vs target (development data)
cs = X_dev["Credit_Score"]
display(cs.groupby(y_dev).agg(count="count", min="min", q1=lambda s: s.quantile(.25), median="median",
                              q3=lambda s: s.quantile(.75), max="max", unique="nunique"))

CS_CUTOFF = int(cs[y_dev == 1].max())                   # highest score observed among positives in development
shared_scores = set(cs[y_dev == 0]) & set(cs[y_dev == 1])
print(f"scores shared by both classes: {len(shared_scores)}   |   threshold rule: Credit_Score <= {CS_CUTOFF}")
print(pd.crosstab(y_dev, (cs <= CS_CUTOFF).astype(int), rownames=["actual"], colnames=["rule"]))

,count,min,q1,median,q3,max,unique
Approved_Flag,,,,,,,
0,36362,659,673.0,682.0,692.0,811,134
1,4706,469,642.0,650.0,655.0,658,106


scores shared by both classes: 0   |   threshold rule: Credit_Score <= 658
rule        0     1
actual             
0       36362     0
1           0  4706


In [6]:
# How much signal survives as proxy-like features are removed?  (RF, pooled 5-fold OOF AUC, preprocessing inside the folds)
NO_CS = [f for f in RAW_FEATURES if f != "Credit_Score"]
NON_AUDITED = [f for f in RAW_FEATURES if f not in AUDITED_PROXY_COLUMNS]
AUDITED_NO_CS = [f for f in AUDITED_PROXY_COLUMNS if f != "Credit_Score"]
assert (len(NO_CS), len(NON_AUDITED), len(AUDITED_NO_CS)) == (82, 48, 34)

def cv_auc(cols):
    pipe = Pipeline([("pre", make_preprocessor(X_dev[cols])),
                     ("rf", RandomForestClassifier(n_estimators=200, min_samples_leaf=20, max_features=0.5,
                                                   class_weight="balanced", random_state=SEED, n_jobs=-1))])
    oof = cross_val_predict(pipe, X_dev[cols], y_dev, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            method="predict_proba")[:, 1]
    return roc_auc_score(y_dev, oof)

ladder = {"Credit_Score only (1)": ["Credit_Score"],
          "All except Credit_Score (82)": NO_CS,
          "Audited proxies except Credit_Score (34)": AUDITED_NO_CS,
          "Non-audited only (48)": NON_AUDITED}
display(pd.Series({name: cv_auc(cols) for name, cols in ladder.items()}, name="RF 5-fold CV ROC-AUC").to_frame())

,RF 5-fold CV ROC-AUC
Credit_Score only (1),1.000000
All except Credit_Score (82),0.976564
Audited proxies except Credit_Score (34),0.966453
Non-audited only (48),0.750585


## 5. Feature selection (development data only)

One function is run twice: on all 83 raw features (`WITH_CS`) and on the 82 features without Credit_Score (`NO_CS`).
Per-encoded-column scores are summed into raw-feature scores before ranking, so one-hot dummies and missing indicators never appear as separate features. F1 is therefore *transformed-column MI aggregated to the raw-feature level* (a heuristic that favours features with many descendant columns), not the MI of the raw variable itself.

In [7]:
def seed_ranks(seed, Z, target, sample, names, mapping):
    """Raw-feature SHAP ranks (1 = most important) of one XGBoost model trained with `seed`."""
    model = XGBClassifier(**XGB_SELECTOR_PARAMS, random_state=seed, n_jobs=1).fit(Z, target)
    return to_raw(mean_abs_shap(model, sample), names, mapping).rank(ascending=False, method="average")


def select_features(cols):
    frame = X_dev[cols]
    pre = make_preprocessor(frame).fit(frame)
    Z, names, mapping = pre.transform(frame), pre.get_feature_names_out(), encoded_to_raw(pre)
    target = y_dev.to_numpy()

    # F1 - mutual information
    discrete = np.array([n.startswith("cat__") or "missingindicator_" in n for n in names]) if MI_DISCRETE_MASK else False
    mi = to_raw(mutual_info_classif(Z, target, discrete_features=discrete, random_state=SEED, n_jobs=N_JOBS), names, mapping)

    # F2 - SHAP importance of one XGBoost model
    selector = XGBClassifier(**XGB_SELECTOR_PARAMS, random_state=SEED, n_jobs=-1).fit(Z, target)
    shap_imp = to_raw(mean_abs_shap(selector, Z[:SHAP_SAMPLE]), names, mapping)

    # F3 - SHAP-rank stability across seeds (each model single-threaded => results independent of parallelism)
    ranks = pd.DataFrame(Parallel(n_jobs=N_JOBS)(
        delayed(seed_ranks)(s, Z, target, Z[:STABILITY_SAMPLE], names, mapping) for s in range(N_SEEDS)))
    assert ranks.shape == (N_SEEDS, len(cols)) and ranks.notna().all().all()

    stability = pd.DataFrame({"mean_rank": ranks.mean(), "std_rank": ranks.std(),
                              "min_rank": ranks.min(), "max_rank": ranks.max()})
    stability["rank_range"] = stability["max_rank"] - stability["min_rank"]
    stability["stable"] = (stability["mean_rank"] <= MEAN_RANK_MAX) & (stability["rank_range"] <= RANK_RANGE_MAX)
    stability = stability.sort_values(["mean_rank", "rank_range"]).rename_axis("raw_feature")
    stability.insert(0, "stability_rank", range(1, len(stability) + 1))

    return {"F1_MI": mi.nlargest(TOP_K).index.tolist(),
            "F2_SHAP": shap_imp.nlargest(TOP_K).index.tolist(),
            "F3_STABLE_SHAP": stability.index[stability["stable"]].tolist(),
            "mi": mi, "shap": shap_imp, "stability": stability}


BRANCHES = {"NO_CS": NO_CS}
if RUN_WITH_CREDIT_SCORE_BRANCH:
    BRANCHES["WITH_CS"] = RAW_FEATURES

SELECTION = {name: select_features(cols) for name, cols in BRANCHES.items()}

for name, sel in SELECTION.items():
    print(f"\n=== {name}: {len(BRANCHES[name])} raw features ===")
    for key in ("F1_MI", "F2_SHAP", "F3_STABLE_SHAP"):
        print(f"{key:<15}({len(sel[key]):>2}) {sel[key]}")
    display(sel["stability"].head(TOP_K + 3))

suffix = {"NO_CS": "phase6_no_credit_score", "WITH_CS": "phase3_with_credit_score"}
for name, sel in SELECTION.items():
    sel["stability"].to_csv(OUT / f"{suffix[name]}_shap_stability_summary.csv")


=== NO_CS: 82 raw features ===
F1_MI          (10) ['enq_L3m', 'enq_L6m', 'enq_L12m', 'time_since_recent_enq', 'PL_enq_L6m', 'tot_enq', 'pct_PL_enq_L6m_of_ever', 'PL_enq_L12m', 'pct_PL_enq_L6m_of_L12m', 'PL_enq']
F2_SHAP        (10) ['enq_L3m', 'Age_Oldest_TL', 'num_times_delinquent', 'time_since_recent_enq', 'time_since_recent_deliquency', 'pct_PL_enq_L6m_of_L12m', 'num_std_12mts', 'pct_PL_enq_L6m_of_ever', 'max_deliq_12mts', 'max_recent_level_of_deliq']
F3_STABLE_SHAP ( 9) ['enq_L3m', 'Age_Oldest_TL', 'num_times_delinquent', 'time_since_recent_enq', 'time_since_recent_deliquency', 'pct_PL_enq_L6m_of_L12m', 'pct_PL_enq_L6m_of_ever', 'num_std_12mts', 'max_deliq_12mts']


,stability_rank,mean_rank,std_rank,min_rank,max_rank,rank_range,stable
raw_feature,,,,,,,
enq_L3m,1,1.00,0.000000,1.0,1.0,0.0,True
Age_Oldest_TL,2,2.00,0.000000,2.0,2.0,0.0,True
num_times_delinquent,3,3.02,0.140705,3.0,4.0,1.0,True
time_since_recent_enq,4,4.01,0.224508,3.0,5.0,2.0,True
time_since_recent_deliquency,5,4.97,0.171447,4.0,5.0,1.0,True
pct_PL_enq_L6m_of_L12m,6,6.00,0.000000,6.0,6.0,0.0,True
pct_PL_enq_L6m_of_ever,7,7.21,0.433333,7.0,9.0,2.0,True
num_std_12mts,8,7.82,0.435310,7.0,9.0,2.0,True
max_deliq_12mts,9,9.17,0.377525,9.0,10.0,1.0,True



=== WITH_CS: 83 raw features ===
F1_MI          (10) ['Credit_Score', 'enq_L3m', 'enq_L6m', 'enq_L12m', 'time_since_recent_enq', 'PL_enq_L6m', 'PL_enq_L12m', 'pct_PL_enq_L6m_of_ever', 'tot_enq', 'pct_PL_enq_L6m_of_L12m']
F2_SHAP        (10) ['Credit_Score', 'enq_L3m', 'num_times_delinquent', 'Age_Oldest_TL', 'time_since_recent_enq', 'pct_PL_enq_L6m_of_ever', 'pct_PL_enq_L6m_of_L12m', 'time_since_recent_deliquency', 'max_recent_level_of_deliq', 'enq_L6m']
F3_STABLE_SHAP ( 5) ['Credit_Score', 'enq_L3m', 'num_times_delinquent', 'time_since_recent_enq', 'Age_Oldest_TL']


,stability_rank,mean_rank,std_rank,min_rank,max_rank,rank_range,stable
raw_feature,,,,,,,
Credit_Score,1,1.00,0.000000,1.0,1.0,0.0,True
enq_L3m,2,2.00,0.000000,2.0,2.0,0.0,True
num_times_delinquent,3,3.00,0.000000,3.0,3.0,0.0,True
time_since_recent_enq,4,4.57,0.685418,4.0,7.0,3.0,True
Age_Oldest_TL,5,4.63,0.630135,4.0,7.0,3.0,True
pct_PL_enq_L6m_of_L12m,6,6.81,1.143228,5.0,10.0,5.0,False
pct_PL_enq_L6m_of_ever,7,7.23,1.117853,5.0,10.0,5.0,False
time_since_recent_deliquency,8,7.87,0.895048,6.0,10.0,4.0,False
max_recent_level_of_deliq,9,9.24,1.944274,4.0,18.0,14.0,False


## 6. Frozen model configuration

The parameters in `MODEL_PARAMS` (§3) come from a randomized search on the 48 non-audited features (all-feature tuning is
degenerate: CV AUC = 1.0 because of Credit_Score). They are reused unchanged for every feature set so that feature-set
effects are not confounded with re-tuning. Set `RUN_TUNING = True` to reproduce the search.

In [8]:
TUNING_GRIDS = {
    "RF":   {"n_estimators": [200, 400, 600], "max_depth": [4, 6, 8, None], "min_samples_leaf": [1, 5, 20], "max_features": ["sqrt", 0.5]},
    "XGB":  {"n_estimators": [150, 300, 450], "max_depth": [3, 5, 7], "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]},
    "LGBM": {"n_estimators": [150, 300, 450], "num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]},
}

if RUN_TUNING:
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    for name, grid in TUNING_GRIDS.items():
        fixed = {k: v for k, v in MODEL_PARAMS[name].items() if k not in grid}
        pipe = Pipeline([("pre", make_preprocessor(X_dev[NON_AUDITED])),        # refit inside every CV fold
                         ("model", MODEL_CLASSES[name](**fixed, random_state=SEED, n_jobs=1))])
        search = RandomizedSearchCV(pipe, {f"model__{k}": v for k, v in grid.items()}, n_iter=20, scoring="roc_auc",
                                    cv=cv, random_state=SEED, n_jobs=N_JOBS).fit(X_dev[NON_AUDITED], y_dev)
        print(f"{name}: CV AUC {search.best_score_:.6f}  {search.best_params_}")
else:
    print("RUN_TUNING=False -> using frozen MODEL_PARAMS:")
    for name, params in MODEL_PARAMS.items():
        print(f"  {name}: {params}")

RUN_TUNING=False -> using frozen MODEL_PARAMS:
  RF: {'n_estimators': 600, 'min_samples_leaf': 20, 'max_features': 0.5, 'max_depth': None, 'class_weight': 'balanced'}
  XGB: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 1.0, 'scale_pos_weight': np.float64(7.726731831704208), 'objective': 'binary:logistic', 'eval_metric': 'logloss'}
  LGBM: {'n_estimators': 150, 'num_leaves': 31, 'learning_rate': 0.03, 'subsample': 1.0, 'colsample_bytree': 1.0, 'scale_pos_weight': np.float64(7.726731831704208), 'objective': 'binary', 'verbosity': -1}


## 7. Final held-out evaluation  *(the only place the test set is used)*

Each feature set gets its own preprocessor fitted on development data; each model is fitted on all development rows and scored once on the test set.

In [9]:
def evaluate_feature_sets(feature_sets):
    rows, preds = [], {}
    for set_name, feats in feature_sets.items():
        pre = make_preprocessor(X_dev[feats])
        Z_dev, Z_test = pre.fit_transform(X_dev[feats]), pre.transform(X_test[feats])
        for model_name in MODEL_PARAMS:
            prob = make_model(model_name).fit(Z_dev, y_dev).predict_proba(Z_test)[:, 1]
            rows.append({"Feature_Set": set_name, "Model": model_name, "N_Features": len(feats),
                         **full_metrics(y_test.to_numpy(), prob)})
            preds[f"{set_name}__{model_name}"] = prob
    results = pd.DataFrame(rows)
    return results, pd.DataFrame(preds, index=X_test.index).assign(y_true=y_test.to_numpy())


def feature_sets_for(branch):
    tag = "_NO_CS" if branch == "NO_CS" else ""
    pick = lambda k: FROZEN_SETS[branch][k] if USE_FROZEN_SETS and k in FROZEN_SETS[branch] else SELECTION[branch][k]
    return {f"F0_ALL{tag}": BRANCHES[branch], f"F1_MI{tag}": pick("F1_MI"),
            f"F2_SHAP{tag}": pick("F2_SHAP"), f"F3_STABLE_SHAP{tag}": pick("F3_STABLE_SHAP")}


EVAL = {}
for branch in BRANCHES:
    sets = feature_sets_for(branch)
    assert all(len(f) == len(set(f)) for f in sets.values())
    assert (branch == "WITH_CS") or all("Credit_Score" not in f for f in sets.values())
    EVAL[branch] = evaluate_feature_sets(sets)

    results, preds = EVAL[branch]
    n_all = len(BRANCHES[branch])
    print(f"\n=== {branch} ===")
    print(results.sort_values(["ROC_AUC", "PR_AUC"], ascending=False)[
        ["Feature_Set", "Model", "N_Features", "ROC_AUC", "PR_AUC", "KS", "Precision", "Recall", "F1", "F2"]
    ].to_string(index=False, float_format="{:.6f}".format))
    print("\nROC-AUC:")
    print(results.pivot(index="Feature_Set", columns="Model", values="ROC_AUC").to_string(float_format="{:.6f}".format))
    print("\nFeature reduction vs F0:")
    print(pd.Series({k: 1 - len(v) / n_all for k, v in sets.items()}).map("{:.2%}".format).to_string())

# save (file names kept from the original project)
res_no_cs, pred_no_cs = EVAL["NO_CS"]
res_no_cs.to_csv(OUT / "phase6_no_credit_score_final_test_results.csv", index=False)
pred_no_cs.to_csv(OUT / "phase6_no_credit_score_test_predictions.csv")
if "WITH_CS" in EVAL:
    EVAL["WITH_CS"][0].to_csv(OUT / "phase3_with_credit_score_final_test_results.csv", index=False)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== NO_CS ===
         Feature_Set Model  N_Features  ROC_AUC   PR_AUC       KS  Precision   Recall       F1       F2
        F0_ALL_NO_CS   XGB          82 0.978732 0.863788 0.864892   0.551607 0.963435 0.701548 0.838266
       F2_SHAP_NO_CS   XGB          10 0.977912 0.860436 0.859079   0.538936 0.965136 0.691651 0.833333
F3_STABLE_SHAP_NO_CS   XGB           9 0.977866 0.860175 0.857129   0.537512 0.962585 0.689823 0.831131
        F0_ALL_NO_CS  LGBM          82 0.977851 0.854561 0.858669   0.528346 0.966837 0.683293 0.829201
       F2_SHAP_NO_CS  LGBM          10 0.977246 0.852590 0.853376   0.522622 0.962585 0.677439 0.823872
F3_STABLE_SHAP_NO_CS  LGBM           9 0.977180 0.851840 0.852239   0.522079 0.965136 0.677612 0.825095
F3_STABLE_SHAP_NO_CS    RF           9 0.974831 0.841321 0.851372   0.526119 0.959184 0.679518 0.823598
        F0_ALL_NO_CS    RF          82 0.974745 0.841466 0.852802   0.542233 0.949830 0.690358 0.825695
       F2_SHAP_NO_CS    RF          10 0.974677 0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



=== WITH_CS ===
   Feature_Set Model  N_Features  ROC_AUC   PR_AUC       KS  Precision   Recall       F1       F2
        F0_ALL   XGB          83 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
        F0_ALL  LGBM          83 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
         F1_MI   XGB          10 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
         F1_MI  LGBM          10 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
       F2_SHAP   XGB          10 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
       F2_SHAP  LGBM          10 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
F3_STABLE_SHAP   XGB           5 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
F3_STABLE_SHAP  LGBM           5 0.999890 0.998302 0.999780   0.998302 1.000000 0.999150 0.999660
       F2_SHAP    RF          10 0.999846 0.997449 0.999780   0.998302 1.000000 0.999150 0.999660
   

In [10]:
# Credit_Score on the test set (diagnostic; the cutoff was derived from development data)
rule = (X_test["Credit_Score"] <= CS_CUTOFF).astype(int)
print(f"Credit_Score-only test ROC-AUC : {roc_auc_score(y_test, -X_test['Credit_Score']):.6f}")
print(f"threshold-rule exceptions      : {(rule != y_test).sum()} / {len(y_test)}  (score, actual): "
      f"{list(zip(X_test.loc[rule != y_test, 'Credit_Score'], y_test[rule != y_test]))}")

Credit_Score-only test ROC-AUC : 0.999780
threshold-rule exceptions      : 2 / 10268  (score, actual): [(489, 0), (499, 0)]


## 8. Paired bootstrap (NO_CS branch)

All models are resampled with the **same** test-row indices (same generator seed), so every difference is paired.
Intervals are unadjusted percentile 95 % CIs; with 24 metric/model/comparison cells, isolated "excludes zero" flags should not be read as findings.

In [11]:
BOOT_COMPARISONS = [("F0_ALL_NO_CS", "F3_STABLE_SHAP_NO_CS", "F3_vs_F0"),
                    ("F2_SHAP_NO_CS", "F3_STABLE_SHAP_NO_CS", "F3_vs_F2")]
y_arr = y_test.to_numpy()

def bootstrap_metrics(prob):
    rng = np.random.default_rng(SEED)                   # identical resamples for every column
    draws = {k: np.empty(N_BOOT) for k in ("ROC_AUC", "PR_AUC", "Recall", "F2")}
    for b in range(N_BOOT):
        idx = rng.integers(0, len(y_arr), size=len(y_arr))
        for k, v in core_metrics(y_arr[idx], prob[idx]).items():
            draws[k][b] = v
    return draws

columns = sorted({f"{s}__{m}" for a, b, _ in BOOT_COMPARISONS for s in (a, b) for m in MODEL_PARAMS})
boot = dict(zip(columns, Parallel(n_jobs=N_JOBS)(delayed(bootstrap_metrics)(pred_no_cs[c].to_numpy()) for c in columns)))
observed = {c: core_metrics(y_arr, pred_no_cs[c].to_numpy()) for c in columns}

rows = []
for base, reduced, label in BOOT_COMPARISONS:
    for model in MODEL_PARAMS:
        a, b = f"{base}__{model}", f"{reduced}__{model}"
        for metric in ("ROC_AUC", "PR_AUC", "Recall", "F2"):
            diff = boot[b][metric] - boot[a][metric]
            lo, hi = np.percentile(diff, [2.5, 97.5])
            rows.append({"Comparison": label, "Model": model, "Baseline": base, "Reduced": reduced, "Metric": metric,
                         "A_value": observed[a][metric], "B_value": observed[b][metric],
                         "Difference_B_minus_A": observed[b][metric] - observed[a][metric],
                         "CI_2.5%": lo, "CI_97.5%": hi, "CI_excludes_zero_unadjusted": bool(lo > 0 or hi < 0),
                         "Bootstrap_N": N_BOOT})
bootstrap_results = pd.DataFrame(rows)
bootstrap_results.to_csv(OUT / "phase6_paired_bootstrap_results.csv", index=False)
print(bootstrap_results[["Comparison", "Model", "Metric", "Difference_B_minus_A", "CI_2.5%", "CI_97.5%",
                         "CI_excludes_zero_unadjusted"]].to_string(index=False, float_format="{:.6f}".format))

Comparison Model  Metric  Difference_B_minus_A   CI_2.5%  CI_97.5%  CI_excludes_zero_unadjusted
  F3_vs_F0    RF ROC_AUC              0.000087 -0.000775  0.000948                        False
  F3_vs_F0    RF  PR_AUC             -0.000145 -0.005083  0.004820                        False
  F3_vs_F0    RF  Recall              0.009354  0.000873  0.017905                         True
  F3_vs_F0    RF      F2             -0.002097 -0.009043  0.004884                        False
  F3_vs_F0   XGB ROC_AUC             -0.000867 -0.001648 -0.000134                         True
  F3_vs_F0   XGB  PR_AUC             -0.003613 -0.007651  0.000417                        False
  F3_vs_F0   XGB  Recall             -0.000850 -0.009984  0.008341                        False
  F3_vs_F0   XGB      F2             -0.007135 -0.014431 -0.000019                         True
  F3_vs_F0  LGBM ROC_AUC             -0.000671 -0.001365 -0.000018                         True
  F3_vs_F0  LGBM  PR_AUC             -0.

## 9. Consistency checks and reproducibility manifest

In [12]:
# (a) Overlap between each selected set and the project's earlier proxy audit
overlap = pd.DataFrame([
    {"branch": b, "feature_set": k, "n_features": len(sel[k]),
     "n_in_audited_list": sum(f in AUDITED_PROXY_COLUMNS for f in sel[k]),
     "not_in_audited_list": [f for f in sel[k] if f not in AUDITED_PROXY_COLUMNS]}
    for b, sel in SELECTION.items() for k in ("F1_MI", "F2_SHAP", "F3_STABLE_SHAP")])
display(overlap)

# (b) Re-derived feature sets vs the frozen lists (comparison only)
for b, sets in FROZEN_SETS.items():
    for k, expected in sets.items():
        if b in SELECTION:
            diff = set(SELECTION[b][k]) ^ set(expected)
            print(f"{b:<8}{k:<16}", "re-derived list MATCHES frozen list" if not diff else f"re-derived list DIFFERS from frozen list: {sorted(diff)}")

# (c) Manifest
manifest = {
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "lightgbm": lightgbm.__version__, "shap": shap.__version__},
    "config": {"split_seed": SPLIT_SEED, "seed": SEED, "test_size": TEST_SIZE, "top_k": TOP_K, "n_seeds": N_SEEDS,
               "mean_rank_max": MEAN_RANK_MAX, "rank_range_max": RANK_RANGE_MAX, "shap_sample": SHAP_SAMPLE,
               "stability_sample": STABILITY_SAMPLE, "threshold": THRESHOLD, "n_boot": N_BOOT,
               "scale_pos_weight": float(SPW), "mi_discrete_mask": MI_DISCRETE_MASK, "use_frozen_sets": USE_FROZEN_SETS},
    "model_params": MODEL_PARAMS,
    "selected_features": {b: {k: sel[k] for k in ("F1_MI", "F2_SHAP", "F3_STABLE_SHAP")} for b, sel in SELECTION.items()},
}
json.dump(manifest, open(OUT / "run_manifest.json", "w"), indent=2, default=str)
print("\nSaved outputs in", OUT.resolve())

,branch,feature_set,n_features,n_in_audited_list,not_in_audited_list
0,NO_CS,F1_MI,10,10,[]
1,NO_CS,F2_SHAP,10,9,[Age_Oldest_TL]
2,NO_CS,F3_STABLE_SHAP,9,8,[Age_Oldest_TL]
3,WITH_CS,F1_MI,10,10,[]
4,WITH_CS,F2_SHAP,10,9,[Age_Oldest_TL]
5,WITH_CS,F3_STABLE_SHAP,5,4,[Age_Oldest_TL]


NO_CS   F1_MI            re-derived list MATCHES frozen list
NO_CS   F2_SHAP          re-derived list MATCHES frozen list
NO_CS   F3_STABLE_SHAP   re-derived list MATCHES frozen list
WITH_CS F2_SHAP          re-derived list DIFFERS from frozen list: ['enq_L6m', 'num_std_12mts']
WITH_CS F3_STABLE_SHAP   re-derived list MATCHES frozen list

Saved outputs in /kaggle/working/phase6_optimized_outputs


## 10. Threshold and class-weight optimization (development-only OOF) — Part A + B

**Scope of this section only.** Per the current review round, this implements *only* Part A (decision-threshold
optimization) and Part B (class-weight / `scale_pos_weight` search), run **jointly**, on the frozen `F0_ALL_NO_CS`,
`F2_SHAP_NO_CS` and `F3_STABLE_SHAP_NO_CS` feature sets. Hyperparameter search, calibration, ensembling and any
change to the SHAP-stability rule are explicitly **out of scope** here and are not run.

**Protocol (development data only, until the single final test evaluation at the end):**
1. For every (feature set × model × weight candidate), generate 5-fold `StratifiedKFold` out-of-fold (OOF)
   probabilities on `X_dev` / `y_dev`. Preprocessing (`make_preprocessor`) is **re-fit inside every fold** on that
   fold's training rows only and applied to that fold's validation rows — no fold ever sees its own validation
   rows during `fit`.
2. All other model hyperparameters are kept exactly as frozen in `MODEL_PARAMS` (§3); only the weighting parameter
   is swapped (`scale_pos_weight` for XGB/LGBM, `class_weight` for RF).
3. Each OOF probability vector is swept over thresholds 0.01–0.99 (step 0.01) and six operating points are located
   (F1-max, F2-max, precision-max s.t. recall≥0.80, F1-max s.t. recall≥0.80, F1-max s.t. recall≥0.85, Youden-J-max).
4. **Primary, pre-registered selection rule:** maximize OOF F1 subject to OOF recall ≥ 0.80 (operating point "D").
   This rule — and only this rule — decides both the frozen weight candidate and the frozen threshold. No other
   rule and no test-set number is used to choose between candidates.
5. Once a (feature set, model) pair has a frozen weight + threshold, the model is refit once on **all** of
   `X_dev` (preprocessing refit on all of `X_dev` too) and evaluated **exactly once** on the untouched test set.
6. Outputs are written to `phase6_optimization_ab_outputs/`, a directory separate from both `phase6_outputs/`
   (the original frozen run) and `phase6_optimized_outputs/` (§1–§9 of this notebook).

In [19]:
AB_OUT = Path("phase6_optimization_ab_outputs"); AB_OUT.mkdir(exist_ok=True)   # separate from phase6_outputs/ and phase6_optimized_outputs/

AB_FEATURE_SETS = {k: v for k, v in feature_sets_for("NO_CS").items() if k in ("F0_ALL_NO_CS", "F2_SHAP_NO_CS", "F3_STABLE_SHAP_NO_CS")}
assert len(AB_FEATURE_SETS["F3_STABLE_SHAP_NO_CS"]) == 9, "F3 must remain the frozen 9-feature set"
assert set(AB_FEATURE_SETS["F3_STABLE_SHAP_NO_CS"]) == set(FROZEN_SETS["NO_CS"]["F3_STABLE_SHAP"])

THRESH_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)
N_SPLITS_AB = 5

# Candidate weightings (Part B). RF's "balanced" already ~matches XGB/LGBM scale_pos_weight in effect,
# so a small custom dict is included to probe the middle ground between None and "balanced".
WEIGHT_GRIDS_AB = {
    "XGB":  [1.0, 2.0, 4.0, 6.0, SPW],
    "LGBM": [1.0, 2.0, 4.0, 6.0, SPW],
    "RF":   [None, "balanced", {0: 1, 1: 3}],
}

def weight_label(model_name, w):
    if model_name == "RF":
        return "None" if w is None else (w if isinstance(w, str) else str(w))
    return f"{w:.4f}" if isinstance(w, float) else str(w)


def build_model_ab(model_name, weight_param, seed=SEED, n_jobs=1):
    """Frozen MODEL_PARAMS (§3) with only the weighting parameter swapped.
    n_jobs=1 by default: these are built INSIDE an outer joblib Parallel (over configs/folds), and nesting
    n_jobs=-1 model-level threading inside process-based outer parallelism oversubscribes cores and can hang
    (the same reason §9's XGB_SELECTOR_PARAMS pins n_jobs=1 for the 100-seed SHAP-stability loop). Pass
    n_jobs=-1 explicitly only for a single, non-parallel fit (e.g. the final refit-on-all-development step)."""
    params = dict(MODEL_PARAMS[model_name])
    if model_name == "RF":
        params["class_weight"] = weight_param
    else:
        params["scale_pos_weight"] = weight_param
    return MODEL_CLASSES[model_name](**params, random_state=seed, n_jobs=n_jobs)


def oof_probabilities(cols, model_name, weight_param, n_splits=N_SPLITS_AB, seed=SEED):
    """Pooled OOF probabilities. Preprocessing and the model are both fit on the fold's TRAIN rows only.
    Runs single-threaded (n_jobs=1) because this whole function is itself dispatched inside an outer
    Parallel(n_jobs=N_JOBS) over (feature_set, model, weight) configs — see the note in build_model_ab."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    Xc, yc = X_dev[cols], y_dev.to_numpy()
    oof = np.full(len(yc), np.nan)
    for train_idx, val_idx in skf.split(Xc, yc):
        X_tr, X_val, y_tr = Xc.iloc[train_idx], Xc.iloc[val_idx], yc[train_idx]
        pre = make_preprocessor(X_tr).fit(X_tr)                      # fit on this fold's train rows only
        Z_tr, Z_val = pre.transform(X_tr), pre.transform(X_val)      # val rows only ever transformed, never fit on
        model = build_model_ab(model_name, weight_param, seed=seed, n_jobs=1).fit(Z_tr, y_tr)
        oof[val_idx] = model.predict_proba(Z_val)[:, 1]
    assert not np.isnan(oof).any(), "every development row must receive exactly one OOF prediction"
    return oof


def threshold_sweep(y_true, y_prob, grid=THRESH_GRID):
    rows = []
    for t in grid:
        tn, fp, fn, tp = confusion(y_true, y_prob, thr=t)
        rows.append({"threshold": float(t),
                     "precision": tp / (tp + fp) if (tp + fp) else 0.0,
                     "recall": tp / (tp + fn) if (tp + fn) else 0.0,
                     "f1": f_beta(tp, fp, fn, 1), "f2": f_beta(tp, fp, fn, 2),
                     "pred_positive_rate": (tp + fp) / len(y_true),
                     "tp": tp, "fp": fp, "tn": tn, "fn": fn})
    return pd.DataFrame(rows)


def pick_operating_points(sweep):
    """Six candidate operating points (§Part A, items A-F). Ties broken toward the lower threshold (idxmax default)."""
    out = {"A_max_F1": sweep.loc[sweep["f1"].idxmax()],
           "B_max_F2": sweep.loc[sweep["f2"].idxmax()]}
    r80, r85 = sweep[sweep["recall"] >= 0.80], sweep[sweep["recall"] >= 0.85]
    out["C_max_precision_recall_ge_080"] = r80.loc[r80["precision"].idxmax()] if len(r80) else None
    out["D_max_F1_recall_ge_080"] = r80.loc[r80["f1"].idxmax()] if len(r80) else None
    out["E_max_F1_recall_ge_085"] = r85.loc[r85["f1"].idxmax()] if len(r85) else None
    specificity = sweep["tn"] / (sweep["tn"] + sweep["fp"]).replace(0, np.nan)
    youden = sweep["recall"] + specificity - 1
    out["F_max_youden_J"] = sweep.loc[youden.idxmax()]
    return out


def full_metrics_at(y_true, y_prob, thr):
    """Same metric definitions as full_metrics() in §3, parameterized by an explicit threshold."""
    tn, fp, fn, tp = confusion(y_true, y_prob, thr=thr)
    return {"ROC_AUC": roc_auc_score(y_true, y_prob), "PR_AUC": average_precision_score(y_true, y_prob),
            "Precision": tp / (tp + fp) if (tp + fp) else 0.0, "Recall": tp / (tp + fn) if (tp + fn) else 0.0,
            "F1": f_beta(tp, fp, fn, 1), "F2": f_beta(tp, fp, fn, 2),
            "KS": ks_2samp(y_prob[y_true == 1], y_prob[y_true == 0]).statistic,
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

print("AB_FEATURE_SETS:", {k: len(v) for k, v in AB_FEATURE_SETS.items()})
print("WEIGHT_GRIDS_AB:", {k: [weight_label(k, w) for w in v] for k, v in WEIGHT_GRIDS_AB.items()})
print(f"Threshold grid: {THRESH_GRID[0]}..{THRESH_GRID[-1]} step 0.01 ({len(THRESH_GRID)} points),  {N_SPLITS_AB}-fold OOF")

AB_FEATURE_SETS: {'F0_ALL_NO_CS': 82, 'F2_SHAP_NO_CS': 10, 'F3_STABLE_SHAP_NO_CS': 9}
WEIGHT_GRIDS_AB: {'XGB': ['1.0000', '2.0000', '4.0000', '6.0000', '7.7267'], 'LGBM': ['1.0000', '2.0000', '4.0000', '6.0000', '7.7267'], 'RF': ['None', 'balanced', '{0: 1, 1: 3}']}
Threshold grid: 0.01..0.99 step 0.01 (99 points),  5-fold OOF


In [ ]:
# Development-only OOF sweep across every (feature set x model x weight candidate).
# This cell never touches X_test / y_test.

def run_one(fs_name, cols, model_name, weight_param):
    oof = oof_probabilities(cols, model_name, weight_param)
    sweep = threshold_sweep(y_dev.to_numpy(), oof)
    points = pick_operating_points(sweep)
    pooled_roc = roc_auc_score(y_dev, oof)
    pooled_pr = average_precision_score(y_dev, oof)
    return oof, sweep, points, pooled_roc, pooled_pr

jobs = [(fs_name, cols, model_name, w)
        for fs_name, cols in AB_FEATURE_SETS.items()
        for model_name in MODEL_PARAMS
        for w in WEIGHT_GRIDS_AB[model_name]]

results = Parallel(n_jobs=N_JOBS)(delayed(run_one)(fs, cols, m, w) for fs, cols, m, w in jobs)

oof_store, sweep_store, point_rows = {}, {}, []
for (fs_name, cols, model_name, w), (oof, sweep, points, pooled_roc, pooled_pr) in zip(jobs, results):
    key = (fs_name, model_name, weight_label(model_name, w))
    oof_store[key] = oof
    sweep_store[key] = sweep
    for op_name, row in points.items():
        point_rows.append({
            "Feature_Set": fs_name, "Model": model_name, "Weight": weight_label(model_name, w),
            "Operating_Point": op_name,
            "Threshold": np.nan if row is None else row["threshold"],
            "OOF_Precision": np.nan if row is None else row["precision"],
            "OOF_Recall": np.nan if row is None else row["recall"],
            "OOF_F1": np.nan if row is None else row["f1"],
            "OOF_F2": np.nan if row is None else row["f2"],
            "OOF_Pred_Positive_Rate": np.nan if row is None else row["pred_positive_rate"],
            "OOF_ROC_AUC_pooled": pooled_roc, "OOF_PR_AUC_pooled": pooled_pr,
        })

operating_points_ab = pd.DataFrame(point_rows)
operating_points_ab.to_csv(AB_OUT / "table_C_threshold_operating_points_all_configs.csv", index=False)
print(f"{len(operating_points_ab)} rows across {len(jobs)} (feature_set, model, weight) configs x 6 operating points")
print(operating_points_ab[operating_points_ab["Operating_Point"] == "D_max_F1_recall_ge_080"]
      .sort_values(["Feature_Set", "Model", "Weight"])
      .to_string(index=False, float_format="{:.6f}".format))

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

In [15]:
# Freeze one weight + one threshold per (feature set, model) using ONLY the pre-registered primary rule:
#     maximize OOF F1 subject to OOF recall >= 0.80   (operating point "D")
# No other rule and no test-set number is consulted here.

PRIMARY_RULE = "D_max_F1_recall_ge_080"
primary = operating_points_ab[operating_points_ab["Operating_Point"] == PRIMARY_RULE].copy()

frozen_ab = []
for (fs_name, model_name), grp in primary.groupby(["Feature_Set", "Model"], sort=False):
    feasible = grp.dropna(subset=["OOF_F1"])
    if feasible.empty:
        # No weight candidate reached recall >= 0.80 anywhere on the grid for this config: fall back to plain F1-max (rule A).
        fallback = operating_points_ab[(operating_points_ab.Feature_Set == fs_name) &
                                        (operating_points_ab.Model == model_name) &
                                        (operating_points_ab.Operating_Point == "A_max_F1")].dropna(subset=["OOF_F1"])
        best = fallback.loc[fallback["OOF_F1"].idxmax()]
        rule_used = "A_max_F1 (fallback: no candidate reached recall>=0.80 under D)"
    else:
        best = feasible.loc[feasible["OOF_F1"].idxmax()]
        rule_used = PRIMARY_RULE
    frozen_ab.append({"Feature_Set": fs_name, "Model": model_name, "Weight": best["Weight"],
                       "Threshold": best["Threshold"], "Selection_Rule": rule_used,
                       "OOF_ROC_AUC_pooled": best["OOF_ROC_AUC_pooled"], "OOF_PR_AUC_pooled": best["OOF_PR_AUC_pooled"],
                       "OOF_Precision": best["OOF_Precision"], "OOF_Recall": best["OOF_Recall"],
                       "OOF_F1": best["OOF_F1"], "OOF_F2": best["OOF_F2"]})

frozen_ab = pd.DataFrame(frozen_ab)
frozen_ab.to_csv(AB_OUT / "table_B_weight_and_threshold_selection.csv", index=False)
print("Frozen (feature_set, model) -> weight + threshold, chosen from development OOF only:\n")
print(frozen_ab.to_string(index=False, float_format="{:.6f}".format))

# --- quality checks (structural, not data-dependent) ---------------------------------------------------
# Note: X_test / y_test are not referenced anywhere in this cell or the two cells above it.
assert set(AB_FEATURE_SETS["F3_STABLE_SHAP_NO_CS"]) == set(FROZEN_SETS["NO_CS"]["F3_STABLE_SHAP"]) and \
       len(AB_FEATURE_SETS["F3_STABLE_SHAP_NO_CS"]) == 9
assert len(frozen_ab) == len(AB_FEATURE_SETS) * len(MODEL_PARAMS)
print("\nQuality checks:")
print("  - OOF probabilities: each fold's preprocessor and model were fit on that fold's train rows only (oof_probabilities()).")
print("  - Threshold selection: chosen only from development OOF sweeps (table_C); test set not referenced above this cell.")
print("  - Weight selection: chosen only via the pre-registered rule D on development OOF; test set not referenced.")
print("  - Feature sets: F0/F2/F3 are unchanged from the frozen §5/§7 lists; F3 remains the 9-feature set (89.02% reduction vs 82).")

NameError: name 'operating_points_ab' is not defined

In [ ]:
# Final, single test-set evaluation. Everything above this line used development data only.
# Each frozen (feature_set, model) config is refit on ALL development rows, then scored once on test.

def refit_and_evaluate(fs_name, cols, model_name, weight_param, thr):
    pre = make_preprocessor(X_dev[cols]).fit(X_dev[cols])                # fit on ALL development rows
    Z_dev, Z_test = pre.transform(X_dev[cols]), pre.transform(X_test[cols])
    model = build_model_ab(model_name, weight_param, seed=SEED).fit(Z_dev, y_dev)
    prob_test = model.predict_proba(Z_test)[:, 1]
    metrics = full_metrics_at(y_test.to_numpy(), prob_test, thr)
    return metrics, prob_test

WEIGHT_LOOKUP_AB = {(k, weight_label(k, w)): w for k, ws in WEIGHT_GRIDS_AB.items() for w in ws}

optimized_rows, optimized_preds = [], {}
for _, row in frozen_ab.iterrows():
    fs_name, model_name = row["Feature_Set"], row["Model"]
    weight_param, thr = WEIGHT_LOOKUP_AB[(model_name, row["Weight"])], row["Threshold"]
    metrics, prob_test = refit_and_evaluate(fs_name, AB_FEATURE_SETS[fs_name], model_name, weight_param, thr)
    optimized_rows.append({"Feature_Set": fs_name, "Model": model_name, "N_Features": len(AB_FEATURE_SETS[fs_name]),
                           "Config": "optimized_dev_only", "Weight": row["Weight"], "Threshold": thr,
                           "Selection_Rule": row["Selection_Rule"], **metrics})
    optimized_preds[f"{fs_name}__{model_name}__optimized"] = prob_test

table_A_optimized = pd.DataFrame(optimized_rows)

# Frozen 0.40-threshold baseline for the same (feature_set, model) cells, already computed in §7 (`res_no_cs`).
baseline = res_no_cs[res_no_cs["Feature_Set"].isin(AB_FEATURE_SETS.keys())].copy()
baseline["Config"], baseline["Weight"], baseline["Threshold"] = "frozen_0.40", "frozen (SPW / balanced)", THRESHOLD
baseline["Selection_Rule"] = "none (fixed threshold, frozen weight)"

TABLE_A_COLS = ["Feature_Set", "Model", "N_Features", "Config", "Weight", "Threshold", "Selection_Rule",
                "ROC_AUC", "PR_AUC", "KS", "Precision", "Recall", "F1", "F2", "TP", "FP", "TN", "FN"]
table_A = pd.concat([baseline[TABLE_A_COLS], table_A_optimized[TABLE_A_COLS]], ignore_index=True)
table_A = table_A.sort_values(["Feature_Set", "Model", "Config"])
table_A.to_csv(AB_OUT / "table_A_current_vs_optimized.csv", index=False)

print("TABLE A — current (frozen 0.40 threshold) vs optimized (development-only threshold + weight):\n")
print(table_A.to_string(index=False, float_format="{:.6f}".format))

# Per-model, per-feature-set delta (optimized minus frozen) for the headline metrics
delta = (table_A.pivot_table(index=["Feature_Set", "Model"], columns="Config",
                             values=["Precision", "Recall", "F1", "F2", "ROC_AUC", "PR_AUC"])
         .pipe(lambda d: d["optimized_dev_only"] - d["frozen_0.40"]))
delta.columns = [f"delta_{c}" for c in delta.columns]
delta.to_csv(AB_OUT / "table_A_deltas.csv")
print("\nDelta (optimized - frozen_0.40):\n")
print(delta.to_string(float_format="{:+.6f}".format))

In [ ]:
# TABLE D — feature reduction (unchanged from §7; restated here for the write-up)
table_D = pd.Series({fs: 1 - len(cols) / len(BRANCHES["NO_CS"]) for fs, cols in AB_FEATURE_SETS.items()},
                    name="reduction_vs_F0").to_frame()
table_D["n_features"] = [len(AB_FEATURE_SETS[fs]) for fs in table_D.index]
table_D["reduction_vs_F0"] = table_D["reduction_vs_F0"].map("{:.2%}".format)
table_D.to_csv(AB_OUT / "table_D_feature_reduction.csv")
print("TABLE D — feature reduction (unchanged):\n")
print(table_D.to_string())

# TABLE E — calibration / ensemble: not run in this round (explicitly out of scope for Part A+B).
table_E_note = "Not run: calibration and ensembling are deferred to a later, separate review round."
print(f"\nTABLE E — {table_E_note}")

manifest_ab = {
    "scope": "Part A (threshold optimization) + Part B (class-weight/scale_pos_weight optimization) only",
    "primary_selection_rule": "maximize OOF F1 subject to OOF recall >= 0.80 (fallback: plain OOF F1-max)",
    "n_splits_oof": N_SPLITS_AB, "threshold_grid": "0.01-0.99 step 0.01",
    "weight_grids": {k: [weight_label(k, w) for w in v] for k, v in WEIGHT_GRIDS_AB.items()},
    "feature_sets": {k: v for k, v in AB_FEATURE_SETS.items()},
    "frozen_selection": frozen_ab.to_dict(orient="records"),
    "out_of_scope_this_round": ["hyperparameter tuning", "probability calibration", "ensembling",
                                "changing the SHAP-stability threshold rule"],
}
json.dump(manifest_ab, open(AB_OUT / "run_manifest_ab.json", "w"), indent=2, default=str)
print("\nSaved outputs in", AB_OUT.resolve())

### Review questions (Part A + B) — fill in from `table_A_current_vs_optimized.csv` and `table_B_weight_and_threshold_selection.csv` after running the cells above

*This notebook was edited and reviewed without execution — the source dataset is fetched via `kagglehub`, which
this editing environment cannot reach. Every cell above is new and has not been run. Run §1–§9 followed by this
section top-to-bottom once, then replace the bracketed placeholders below with the actual numbers before treating
this as a result.*

1. **Did threshold optimization improve precision?** [fill in from `table_A_deltas.csv`, `delta_Precision`]
2. **Did it improve F1?** [fill in, `delta_F1`]
3. **What happened to recall?** [fill in, `delta_Recall` — expected to fall from ~0.96 toward ~0.80–0.85 by construction of the primary rule]
4. **Did ROC-AUC / PR-AUC remain unchanged, as expected?** [fill in, `delta_ROC_AUC` / `delta_PR_AUC` — these are threshold-invariant, so any change should be attributable only to the weight-parameter swap, not to thresholding itself]
5. **Did changing class weighting improve the development tradeoff?** [fill in from `table_B_weight_and_threshold_selection.csv` — compare the chosen `Weight` per model against the frozen SPW/`"balanced"` baseline]
6. **What exact configuration should become the candidate final model?** [fill in — read off the `Feature_Set` / `Model` / `Weight` / `Threshold` row in `table_A_current_vs_optimized.csv` with the best test F1 among `F3_STABLE_SHAP_NO_CS` rows, since F3 remains the proposed method]
7. **Does F3 still retain its 89.02% feature reduction?** Yes — `AB_FEATURE_SETS["F3_STABLE_SHAP_NO_CS"]` is asserted equal to the frozen 9-feature `FROZEN_SETS["NO_CS"]["F3_STABLE_SHAP"]` list; nothing in this section can change it.
8. **Did anything require changing the research methodology?** No — the feature-selection pipeline (§5) and feature sets are untouched; this section only adds a decision-threshold and class-weight layer on top of the existing frozen models, using development-only OOF evidence.

## Notes for the write-up

* **Proxy overlap (§9a).** Most features chosen by F1/F2/F3 are members of the project's own audited-proxy list. Removing `Credit_Score` therefore does not remove the target-proxy family; the compact subset largely *concentrates* it. The 48 non-audited features reach only ≈0.75 CV AUC (§4). Frame the result as compactness/retention, not as selection of "reliable" features.
* **Test-set history.** The test set was inspected during Credit_Score diagnosis (before the `NO_CS` branch existed). Selection and tuning never used it, but the decision to add the `NO_CS` branch was informed by it; disclose this.
* **Frozen parameters** come from tuning on the 48 non-audited features and are not re-tuned per feature set (by design).
* **SHAP is computed on development rows the models were trained on**, so stability describes the fitted models, not out-of-sample explanation stability.
* **MI baseline.** F1 aggregates transformed-column MI; with `MI_DISCRETE_MASK=False` (frozen setting) binary dummies/indicators are treated as continuous. Re-run with `True` and report the sensitivity before claiming MI is a weak baseline.
* **Multiplicity.** Bootstrap intervals are unadjusted; 24 cells were examined.